# ViSER — Training on Kaggle
**Repo:** https://github.com/Huu2412/M-ViSER

**Pipeline:**
1. Clone source code from GitHub
2. Install dependencies
3. Configure paths for Kaggle
4. Train on `AbstractTTS/IEMOCAP` (auto-loaded from HuggingFace)

> Enable **GPU T4 x2** in Settings → Accelerator

## Cell 1 — Clone from GitHub & Install dependencies

In [ ]:
import os, sys

REPO_URL = 'https://github.com/Huu2412/M-ViSER.git'
SOURCE_CODE_PATH = '/kaggle/working/M-ViSER'
WORKING_DIR = '/kaggle/working'

# Clone repo
if not os.path.exists(SOURCE_CODE_PATH):
    !git clone {REPO_URL} {SOURCE_CODE_PATH}
else:
    print('Repo already cloned. Pulling latest...')
    !git -C {SOURCE_CODE_PATH} pull

print('\nRepo contents:')
print([f for f in os.listdir(SOURCE_CODE_PATH) if not f.startswith('.')])

In [ ]:
!nvidia-smi

# Install all dependencies
!pip install -q -r {SOURCE_CODE_PATH}/requirements.txt
!pip install -q torchcodec   # Required for decoding IEMOCAP audio

print('All dependencies installed.')

## Cell 2 — Add repo to Python path

In [ ]:
sys.path.insert(0, SOURCE_CODE_PATH)
os.chdir(SOURCE_CODE_PATH)
print(f'Working dir: {os.getcwd()}')

## Cell 3 — Configure training

In [ ]:
import yaml

# ============================================================
# Adjust these settings as needed
# ============================================================
FOLD        = 1     # Validation fold (1-5)
EPOCHS      = 50
BATCH_SIZE  = 16    # T4 GPU: 16-32
GRAD_ACCUM  = 2
NUM_WORKERS = 2
USE_WANDB   = False
# ============================================================

config_src = os.path.join(SOURCE_CODE_PATH, 'config', 'config.yaml')
config_dst = os.path.join(WORKING_DIR, 'config.yaml')

with open(config_src, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

# Redirect outputs to writable /kaggle/working/
cfg['paths']['output_dir'] = os.path.join(WORKING_DIR, 'checkpoints')
cfg['paths']['cache_dir']  = os.path.join(WORKING_DIR, 'cache_ser')
cfg['paths']['log_dir']    = os.path.join(WORKING_DIR, 'logs_ser')

# Dataset (loaded directly from HuggingFace — no CSV needed)
cfg['dataset']['hf_dataset']   = 'AbstractTTS/IEMOCAP'
cfg['dataset']['current_fold'] = FOLD

# Training
cfg['training']['epochs']                      = EPOCHS
cfg['training']['batch_size']                  = BATCH_SIZE
cfg['training']['gradient_accumulation_steps'] = GRAD_ACCUM
cfg['training']['num_workers']                 = NUM_WORKERS
cfg['logging']['use_wandb']                    = USE_WANDB

with open(config_dst, 'w', encoding='utf-8') as f:
    yaml.dump(cfg, f, allow_unicode=True, default_flow_style=False, sort_keys=False)

print(f'Config saved: {config_dst}')
print(f'  dataset    : {cfg["dataset"]["hf_dataset"]}')
print(f'  fold       : {FOLD}/5')
print(f'  epochs     : {EPOCHS}')
print(f'  batch_size : {BATCH_SIZE}')
print(f'  output_dir : {cfg["paths"]["output_dir"]}')

## Cell 4 — Smoke test (verify pipeline before training)

In [ ]:
print('Running smoke test (model init + fake forward + loss + backward)...')
!python {SOURCE_CODE_PATH}/smoke_test.py
print('If no errors above -> pipeline is ready.')

## Cell 5 — Training

IEMOCAP is loaded automatically from HuggingFace and split into 5 speaker-independent folds.  
Best checkpoint (by `macro_f1` on val set) is saved to `checkpoints/best_model.pt`.

Log format:
```
Epoch 01 | Train [L:2.40 L_emo:1.21 L_ctc:0.12 A:54.3%] | Val [L:1.21 A:65.4% mF1:64.2%] | Time: 120s
```

In [ ]:
!python {SOURCE_CODE_PATH}/train.py --config {config_dst}

## Cell 6 — Show saved checkpoints

In [ ]:
import glob

print('Checkpoints saved:')
ckpts = sorted(glob.glob(os.path.join(WORKING_DIR, 'checkpoints', '**', '*.pt'), recursive=True))
for f in ckpts:
    print(f'  {os.path.basename(f)}  ({os.path.getsize(f)/1e6:.0f} MB)')

best = os.path.join(WORKING_DIR, 'checkpoints', 'best_model.pt')
if os.path.exists(best):
    import torch
    ckpt = torch.load(best, map_location='cpu', weights_only=False)
    print(f'\nBest model:')
    print(f'  Epoch   : {ckpt["epoch"]}')
    for k, v in ckpt.get('val_metrics', {}).items():
        print(f'  {k:<20}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

---
## (Bonus) 5-Fold Cross Validation
Set `RUN_5FOLD = True` to train all 5 folds.

In [ ]:
RUN_5FOLD = False

if RUN_5FOLD:
    !python {SOURCE_CODE_PATH}/run_5fold.py \
        --config {config_src} \
        --output_dir {os.path.join(WORKING_DIR, 'checkpoints_5fold')}
else:
    print('5-Fold skipped. Set RUN_5FOLD = True to enable.')